In [1]:
import pandas as pd
from pathlib import Path

PARQUET = '/workspaces/super-duper-dollop/data/processed/test_dataset.parquet'

df = pd.read_parquet(PARQUET)
print(f'Размер: {df.shape}')
print(f'Колонки: {df.columns.tolist()}')
df.head(10)

Размер: (79, 3)
Колонки: ['Path', 'Label', 'Speaker']


,Path,Label,Speaker
0,/workspaces/super-duper-dollop/data/raw/Sound3...,False,12
1,/workspaces/super-duper-dollop/data/raw/Sound3...,False,12
2,/workspaces/super-duper-dollop/data/raw/Sound3...,False,12
3,/workspaces/super-duper-dollop/data/raw/Sound3...,False,54
4,/workspaces/super-duper-dollop/data/raw/Sound3...,False,54
5,/workspaces/super-duper-dollop/data/raw/Sound3...,False,54
6,/workspaces/super-duper-dollop/data/raw/Sound3...,False,54
7,/workspaces/super-duper-dollop/data/raw/Sound3...,False,54
8,/workspaces/super-duper-dollop/data/raw/Sound3...,False,54
9,/workspaces/super-duper-dollop/data/raw/Sound3...,False,54


In [2]:
print(df.iloc[:, 1].value_counts().rename({0: 'До операции (0)', 1: 'После операции (1)'}).to_string())

Label
После операции (1)    51
До операции (0)       28


In [3]:
def get_speaker(path: str) -> str:
    for part in Path(path).parts:
        if part.isdigit():
            return part
    return 'unknown'

path_col, label_col = df.columns[0], df.columns[1]
df['speaker'] = df[path_col].apply(get_speaker)

summary = (
    df.groupby('speaker')
    .agg(
        total=(label_col, 'count'),
        before=(label_col, lambda x: (x == 0).sum()),
        after=(label_col, lambda x: (x == 1).sum()),
    )
)
print(summary.to_string())

         total  before  after
speaker                      
12           3       3      0
54          76      25     51


In [4]:
def get_session(path: str) -> str:
    parts = Path(path).parts

    return parts[-2] if len(parts) >= 2 else 'unknown'

df['session'] = df[path_col].apply(get_session)

session_summary = (
    df.groupby(['speaker', 'session'])
    .agg(count=(label_col, 'count'), label=(label_col, 'first'))
    .rename(columns={'label': 'класс'})
)
session_summary['класс'] = session_summary['класс'].map({0: 'до', 1: 'после'})
print(session_summary.to_string())

                    count класс
speaker session                
12      12.04.2013      3   NaN
54      05.05.2016     25   NaN
        19.01.2017     26   NaN
        21.03.2016     25   NaN
